# Clase 7: APIs + LLMs — Analizando comentarios de YouTube
## México vs Ecuador, Mundial 2026

En esta clase vamos a:
1. Descargar comentarios de YouTube usando la YouTube Data API v3
2. Limpiarlos y organizarlos con pandas
3. Clasificarlos automáticamente con un LLM de Hugging Face (zero-shot)
4. Visualizar los resultados

La parte más importante: **el mismo código funciona para cualquier video, cualquier tema, cualquier idioma**. No entrenamos nada — el modelo ya sabe clasificar texto sin haber visto ejemplos de fútbol.

---
## Paso 0 — Instalar librerías

Si no las tienes instaladas, corre esto una sola vez en la terminal:

In [ ]:
%%capture
pip install google-api-python-client huggingface_hub pandas matplotlib

In [ ]:
# Importamos todo lo que vamos a necesitar
# Es buena práctica poner todos los imports al inicio del notebook

import pandas as pd                          # manipulación de datos — el tidyverse de Python
import matplotlib.pyplot as plt             # gráficas
from googleapiclient.discovery import build  # cliente oficial de Google APIs
from huggingface_hub import InferenceClient  # cliente de Hugging Face
import time                                  # para pausas entre peticiones
from google.colab import userdata
from transformers import pipeline
import time

---
## Paso 1 — Conseguir las API keys

### YouTube Data API v3
1. Ve a [console.cloud.google.com](https://console.cloud.google.com)
2. Crea un proyecto nuevo (botón arriba a la izquierda)
3. En el menú: **APIs y servicios → Biblioteca**
4. Busca "YouTube Data API v3" → Habilitar
5. **APIs y servicios → Credenciales → Crear credenciales → Clave de API**
6. Copia la key — se ve así: `AIzaSy...`

La cuota gratuita es 10,000 unidades por día. Descargar 100 comentarios cuesta ~3 unidades — no te vas a quedar sin cuota.

### Hugging Face
1. Crea cuenta en [huggingface.co](https://huggingface.co)
2. Ve a **Settings → Access Tokens → New token**
3. Copia el token — se ve así: `hf_...`

El tier gratuito da acceso a modelos de clasificación de texto sin límite estricto para uso educativo.

In [ ]:
# En Google Colab, las keys se guardan en "Secrets" (el ícono del candado en la barra izquierda)
# Así nunca aparecen en el código y no hay riesgo de subirlas a GitHub por accidente
#
# Para agregar un secret:
# 1. Haz clic en el ícono 🔑 en la barra izquierda
# 2. Clic en "Add new secret"
# 3. Name: YOUTUBE_API_KEY  →  Value: tu key de Google Cloud
# 4. Name: HF_TOKEN         →  Value: tu token de Hugging Face
# 5. Activa el toggle "Notebook access" para cada uno

from google.colab import userdata

YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY')
HF_TOKEN        = userdata.get('HF_TOKEN')

# El ID del video está en la URL después de ?v=
# https://www.youtube.com/watch?v=iCFwyDKzihA
#                                  ↑ este es el ID
VIDEO_ID = "iCFwyDKzihA"

print("Keys cargadas correctamente ✓")

---
## Paso 2 — Descargar comentarios de YouTube

### ¿Qué es una API?

Una API (Application Programming Interface) es un servicio que un servidor ofrece para que otros programas puedan pedirle datos de forma estructurada. En vez de scrapear el HTML de YouTube (que es complicado y puede romperse), usamos la API oficial que Google ofrece.

El flujo es siempre el mismo:
1. **Construir** la petición (qué quieres, con qué parámetros)
2. **Ejecutar** la petición (mandarla al servidor)
3. **Parsear** la respuesta (extraer los datos del JSON que devuelve)

In [ ]:
# build() crea el cliente de la API de YouTube
# 'youtube'  = nombre del servicio
# 'v3'       = versión de la API
# developerKey = nuestra key para autenticarnos

youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

print("Cliente de YouTube creado correctamente")

In [ ]:
def descargar_comentarios(video_id, max_comentarios=200):
    """
    Descarga comentarios de un video de YouTube.

    Parámetros:
        video_id        : ID del video (lo que va después de ?v= en la URL)
        max_comentarios : cuántos comentarios descargar (default: 200)

    Retorna:
        Lista de diccionarios, uno por comentario
    """
    comentarios = []   # lista vacía donde vamos a ir acumulando
    page_token  = None  # YouTube pagina los resultados — esto lleva el control de página

    while len(comentarios) < max_comentarios:

        # ── Construir la petición ──────────────────────────────────────────────
        # commentThreads().list() es el endpoint para comentarios
        # part='snippet' indica que queremos el contenido del comentario
        # videoId es el video que queremos
        # maxResults es el máximo por página (límite de la API: 100)
        # order='relevance' trae los más relevantes primero
        # pageToken permite ir a la siguiente página de resultados

        peticion = youtube.commentThreads().list(
            part       = 'snippet',
            videoId    = video_id,
            maxResults = min(100, max_comentarios - len(comentarios)),
            order      = 'relevance',
            pageToken  = page_token
        )

        # ── Ejecutar la petición ───────────────────────────────────────────────
        respuesta = peticion.execute()

        # ── Parsear la respuesta ───────────────────────────────────────────────
        # respuesta es un diccionario con la estructura:
        # {'items': [{comentario1}, {comentario2}, ...], 'nextPageToken': '...'}
        # navegamos hasta el texto del comentario dentro de la estructura anidada

        for item in respuesta['items']:
            snippet = item['snippet']['topLevelComment']['snippet']

            comentarios.append({
                'texto'     : snippet['textDisplay'],          # el texto del comentario
                'autor'     : snippet['authorDisplayName'],    # nombre del usuario
                'likes'     : snippet['likeCount'],            # likes del comentario
                'fecha'     : snippet['publishedAt'],          # fecha de publicación
                'respuestas': item['snippet']['totalReplyCount']  # número de respuestas
            })

        # ── Paginación ─────────────────────────────────────────────────────────
        # si hay más páginas, guardamos el token para la siguiente iteración
        # si no hay, salimos del while
        page_token = respuesta.get('nextPageToken')
        if not page_token:
            break

    print(f"Descargados {len(comentarios)} comentarios")
    return comentarios


# Descargar los comentarios
comentarios_raw = descargar_comentarios(VIDEO_ID, max_comentarios=200)

---
## Paso 3 — Organizar con pandas

**pandas** es la librería de manipulación de datos de Python. Su estructura central es el **DataFrame** — equivale al data frame de R. La sintaxis es similar a tidyverse pero con algunas diferencias importantes.

In [ ]:
# pd.DataFrame() convierte una lista de diccionarios en un DataFrame
# cada diccionario es una fila, las claves son las columnas
# equivale a data.frame() o tibble() en R

df = pd.DataFrame(comentarios_raw)

print(f"Dimensiones: {df.shape}")   # equivale a dim() en R
print(f"\nPrimeras filas:")
df.head()                            # equivale a head() en R

In [ ]:
# Explorar los tipos de cada columna
# equivale a glimpse() o str() en R
df.dtypes

In [ ]:
# Limpieza básica
# ── 1. Convertir fecha a datetime ─────────────────────────────────────────────
# viene como string '2026-06-30T22:15:03.000Z'
# pd.to_datetime() equivale a as.Date() en R
df['fecha'] = pd.to_datetime(df['fecha'])

# ── 2. Limpiar el texto ───────────────────────────────────────────────────────
# YouTube mete tags HTML en los comentarios (<br>, &amp;, etc.)
# .str.replace() equivale a str_replace() en R
# regex=True permite usar expresiones regulares
df['texto_limpio'] = (
    df['texto']
    .str.replace('<br>', ' ', regex=False)   # saltos de línea → espacio
    .str.replace('&amp;', '&', regex=False)  # entidad HTML → símbolo
    .str.replace('&#39;', "'", regex=False)  # comilla HTML → comilla
    .str.strip()                              # quitar espacios al inicio y final
)

# ── 3. Filtrar comentarios muy cortos ─────────────────────────────────────────
# comentarios de menos de 10 caracteres no aportan información
# .str.len() cuenta caracteres por fila
# df[condicion] filtra filas — equivale a filter() en R
df = df[df['texto_limpio'].str.len() >= 10].reset_index(drop=True)
#                                           ↑ reset_index reordena los índices
#                                             después de filtrar

print(f"Comentarios después de limpiar: {len(df)}")
df[['texto_limpio', 'likes', 'respuestas']].head(10)

In [ ]:
# Estadísticas básicas del dataset
# equivale a summary() en R
print("Estadísticas de likes:")
print(df['likes'].describe())

print(f"\nComentario con más likes:")
# .idxmax() devuelve el índice del valor máximo
idx_max = df['likes'].idxmax()
print(f"'{df.loc[idx_max, 'texto_limpio']}' ({df.loc[idx_max, 'likes']} likes)")

---
## Paso 4 — Zero-Shot Classification con Hugging Face

### ¿Qué es zero-shot classification?

Un modelo de clasificación tradicional necesita ejemplos etiquetados para entrenarse: *"este comentario es positivo", "este es negativo"*. Necesitas miles de ejemplos antes de que el modelo aprenda.

**Zero-shot** significa que el modelo clasifica texto en categorías que **nunca vio durante el entrenamiento**. Le dices las categorías en el momento de la consulta y el modelo usa su comprensión del lenguaje para decidir. Cero ejemplos etiquetados, cero entrenamiento.

El modelo que usamos es `facebook/bart-large-mnli` — entrenado en NLI (Natural Language Inference), que es aprender si una hipótesis se sigue de una premisa. La clasificación zero-shot reformula cada categoría como una hipótesis y elige la más probable.

### Categorías que vamos a usar

Para los comentarios de México vs Ecuador tiene sentido clasificar en:
- **celebración**: euforia por la victoria de México
- **crítica**: análisis negativo del juego o los jugadores  
- **burla**: comentarios burlones hacia Ecuador
- **análisis**: comentarios técnicos sobre el partido
- **spam**: publicidad, enlaces, comentarios irrelevantes

In [ ]:
# cargamos el modelo directamente en la GPU de Colab
# device=0 significa "usa la primera GPU disponible"
# si no hay GPU y pusiste CPU por error, cambia a device=-1
# este paso tarda ~1 minuto la primera vez — descarga el modelo de HF

clasificador = pipeline(
    "zero-shot-classification",
    model  = "facebook/bart-large-mnli",
    device = 0
)

print("Modelo cargado en GPU ✓")

In [ ]:
# verificar que la GPU está activa
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'ninguna'}")

---
## Paso 5 — Clasificar todos los comentarios

Ahora procesamos todos los comentarios del dataset. Como mandamos una petición por comentario, hay que ser cuidadosos con el rate limit — por eso agregamos `time.sleep()` entre peticiones.

In [ ]:
def clasificar_comentario(texto, categorias, cliente, modelo="facebook/bart-large-mnli"):
    """
    Clasifica un comentario en una de las categorías dadas.

    Retorna:
        tuple: (categoría ganadora, score de confianza)
    """
    try:
        resultado = cliente.zero_shot_classification(
            text             = texto,
            candidate_labels = categorias,
            model            = modelo
        )
        # resultado es una lista de objetos — usamos índice [0] para el de mayor score
        return resultado[0].label, resultado[0].score

    except Exception as e:
        print(f"Error: {e}")
        return None, None


# Clasificar todos los comentarios
N = min(100, len(df))
df_muestra = df.head(N).copy()

categorias_asignadas = []
scores_asignados     = []

print(f"Clasificando {N} comentarios...")
print("(puede tardar unos minutos — una petición por comentario)\n")

for i, texto in enumerate(df_muestra['texto_limpio']):

    categoria, score = clasificar_comentario(texto, CATEGORIAS, cliente_hf)
    categorias_asignadas.append(categoria)
    scores_asignados.append(score)

    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{N} procesados...")

    time.sleep(0.5)

df_muestra['categoria'] = categorias_asignadas
df_muestra['score']     = scores_asignados

print(f"\n¡Listo! {N} comentarios clasificados.")
df_muestra[['texto_limpio', 'categoria', 'score']].head(10)

In [ ]:
# Ver ejemplos de cada categoría
print("Ejemplos por categoría:\n")

for cat in CATEGORIAS:
    # filtrar comentarios de esta categoría
    ejemplos = df_muestra[df_muestra['categoria'] == cat]['texto_limpio'].head(2)
    print(f"── {cat.upper()} ──")
    for ej in ejemplos:
        print(f"  • {ej[:100]}")
    print()

---
## Paso 6 — Análisis y visualización

In [ ]:
# ── Distribución de categorías ────────────────────────────────────────────────
# .value_counts() cuenta cuántos hay de cada categoría
# equivale a count() en tidyverse
conteo = df_muestra['categoria'].value_counts()
print("Comentarios por categoría:")
print(conteo)
print(f"\nPorcentajes:")
print((conteo / len(df_muestra) * 100).round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Gráfica 1: distribución de categorías ─────────────────────────────────────
colores = ['#2196F3', '#F44336', '#FF9800', '#4CAF50', '#9E9E9E']

conteo.plot(
    kind  = 'barh',    # barras horizontales
    ax    = axes[0],   # en cuál de los dos paneles dibujamos
    color = colores,
    alpha = 0.85
)
axes[0].set_title('Distribución de comentarios por categoría', fontsize=13)
axes[0].set_xlabel('Número de comentarios')
axes[0].set_ylabel('')

# agregar etiquetas con el número al final de cada barra
for i, v in enumerate(conteo.values):
    axes[0].text(v + 0.3, i, str(v), va='center', fontsize=11)

# ── Gráfica 2: confianza del modelo por categoría ─────────────────────────────
# .groupby().mean() equivale a group_by() %>% summarise(mean()) en R
confianza_por_cat = df_muestra.groupby('categoria')['score'].mean().sort_values()

confianza_por_cat.plot(
    kind  = 'barh',
    ax    = axes[1],
    color = 'steelblue',
    alpha = 0.85
)
axes[1].set_title('Confianza promedio del modelo por categoría', fontsize=13)
axes[1].set_xlabel('Score promedio (0 = inseguro, 1 = muy seguro)')
axes[1].set_ylabel('')
axes[1].set_xlim(0, 1)

for i, v in enumerate(confianza_por_cat.values):
    axes[1].text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=11)

plt.suptitle('México vs Ecuador — Análisis de comentarios YouTube\nWorldCup 2026',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── ¿Los comentarios más likeados son más de celebración? ─────────────────────
# Esto conecta el análisis con una pregunta económica real:
# ¿la gente da más likes al contenido emocional o al analítico?

likes_por_cat = (
    df_muestra
    .groupby('categoria')['likes']
    .agg(['mean', 'sum', 'count'])   # varias estadísticas a la vez
    .round(1)
    .sort_values('mean', ascending=False)
)

likes_por_cat.columns = ['likes_promedio', 'likes_total', 'n_comentarios']
print("Likes por categoría:")
print(likes_por_cat)

In [ ]:
# ── Los 5 comentarios más likeados y su categoría ─────────────────────────────
# .sort_values() equivale a arrange() en R
# .head() equivale a slice_head() en R

top5 = (
    df_muestra
    .sort_values('likes', ascending=False)
    .head(5)[['texto_limpio', 'likes', 'categoria', 'score']]
)

print("Top 5 comentarios más likeados:")
for _, fila in top5.iterrows():
    # .iterrows() itera fila por fila — devuelve (índice, fila)
    print(f"\n[{fila['likes']} likes | {fila['categoria']} | score: {fila['score']:.2f}]")
    print(f"  {fila['texto_limpio'][:120]}")

---
## Paso 7 — El mismo código, cualquier video

Para analizar otro video solo cambias el `VIDEO_ID` y las `CATEGORIAS`. El resto del código es idéntico.

In [ ]:
# Ejemplo: si quisieras analizar reseñas de un restaurante
# solo cambiarías las categorías:

CATEGORIAS_RESTAURANTE = [
    "experiencia positiva",
    "experiencia negativa",
    "pregunta sobre el lugar",
    "recomendación a otros",
    "spam"
]

# O para comentarios de un video de política:
CATEGORIAS_POLITICA = [
    "apoyo al gobierno",
    "crítica al gobierno",
    "análisis neutral",
    "desinformación",
    "spam"
]

# O para reseñas de un producto en Amazon/MercadoLibre:
CATEGORIAS_PRODUCTO = [
    "muy satisfecho",
    "insatisfecho",
    "problema con el envío",
    "relación calidad-precio",
    "spam"
]

print("El modelo funciona igual con cualquier conjunto de categorías.")
print("No necesita reentrenamiento — eso es el poder del zero-shot.")

---
## Resumen

Lo que hicimos en esta clase:

| Paso | Herramienta | Qué aprendimos |
|---|---|---|
| Descargar comentarios | YouTube Data API + `google-api-python-client` | Cómo consumir una API con paginación |
| Organizar los datos | `pandas` | DataFrame, limpieza de texto, filtros |
| Clasificar texto | `huggingface_hub` + `facebook/bart-large-mnli` | Zero-shot classification |
| Analizar resultados | `pandas` + `matplotlib` | groupby, agg, visualización |

**La idea central:** combinando una API de datos y un LLM pre-entrenado, podemos construir un pipeline de análisis de texto completo en menos de 100 líneas de código — sin etiquetar datos, sin entrenar modelos, sin GPU.